# AI-Assisted Parallel Image Processing — Colab setup

Trước khi chạy, chọn **Runtime → Change runtime type → GPU**. Notebook sẽ clone repository, cài dependency, tải dữ liệu có kiểm tra checksum, tạo bộ benchmark và build bản Release bằng OpenMP/CUDA.

In [ ]:
!nvidia-smi
!nvcc --version

In [ ]:
REPOSITORY = 'https://github.com/Chicken20145/ai-assisted-parallel-image-processing.git'
GIT_REF = 'main'

%cd /content
!test -d ai-assisted-parallel-image-processing || git clone --branch {GIT_REF} {REPOSITORY}
%cd /content/ai-assisted-parallel-image-processing
!git fetch origin {GIT_REF}
!git switch {GIT_REF}
!git pull --ff-only origin {GIT_REF}
!bash scripts/setup_colab.sh

## Cập nhật code khi A/B/C làm song song

Sau khi thành viên A push commit mới lên cùng branch, chạy cell dưới để lấy code mới, build tăng dần và test lại. Không cần tải lại dataset hoặc cài lại package trong cùng runtime.

In [ ]:
%cd /content/ai-assisted-parallel-image-processing
!git pull --ff-only origin {GIT_REF}
!bash scripts/build_colab.sh
!bash scripts/test_colab.sh

## Mở Streamlit UI trên Colab

Colab không cho trình duyệt truy cập trực tiếp `localhost:8501`. Cell dưới chạy Streamlit nền, chờ health check và tạo URL qua Colab kernel proxy. Chạy lại cell sẽ dừng server cũ trước khi mở server mới.

In [ ]:
import os
import subprocess
import sys
import time
from pathlib import Path

import requests
from google.colab import output
from IPython.display import HTML, display

repo = Path('/content/ai-assisted-parallel-image-processing')
core_cli = repo / 'build-colab' / 'image_pipeline_cli'
if not core_cli.is_file():
    raise FileNotFoundError('Chưa có image_pipeline_cli. Hãy chạy cell setup/build trước.')

if '_pixel_lab_process' in globals() and _pixel_lab_process.poll() is None:
    _pixel_lab_process.terminate()
    try:
        _pixel_lab_process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        _pixel_lab_process.kill()
if '_pixel_lab_log' in globals() and not _pixel_lab_log.closed:
    _pixel_lab_log.close()

environment = os.environ.copy()
environment['PIP_CORE_CLI'] = str(core_cli)
log_path = Path('/tmp/pixel_lab_streamlit.log')
_pixel_lab_log = log_path.open('w', encoding='utf-8')
_pixel_lab_process = subprocess.Popen(
    [sys.executable, '-m', 'streamlit', 'run', 'app/app.py',
     '--server.address=0.0.0.0', '--server.port=8501',
     '--server.headless=true', '--server.enableCORS=false',
     '--server.enableXsrfProtection=false'],
    cwd=repo, env=environment, stdout=_pixel_lab_log,
    stderr=subprocess.STDOUT,
)

for _ in range(30):
    if _pixel_lab_process.poll() is not None:
        _pixel_lab_log.flush()
        raise RuntimeError(log_path.read_text(encoding='utf-8', errors='replace'))
    try:
        if requests.get('http://127.0.0.1:8501/_stcore/health', timeout=1).ok:
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    _pixel_lab_process.terminate()
    _pixel_lab_log.flush()
    raise TimeoutError(log_path.read_text(encoding='utf-8', errors='replace'))

ui_url = output.eval_js('google.colab.kernel.proxyPort(8501)')
display(HTML(f'<a href="{ui_url}" target="_blank" style="font-size:18px">Mở Pixel Lab Streamlit UI</a>'))
print('Server đang chạy. Log:', log_path)

## Lưu kết quả benchmark lên Drive (tùy chọn)

Chỉ mount Drive khi cần lưu kết quả. Nên benchmark trên ổ đĩa cục bộ `/content` rồi mới chép CSV/biểu đồ sang Drive để I/O mạng không làm sai lệch thời gian.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import shutil

source = Path('/content/ai-assisted-parallel-image-processing/benchmarks/results')
destination = Path('/content/drive/MyDrive/parallel-image-processing/results')
if source.exists():
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source, destination, dirs_exist_ok=True)
    print(f'Copied results to {destination}')
else:
    print('No benchmark results yet.')